In [1]:
!pip install optuna

## Importing the Necessary Libraries and the Dataset

In [2]:
import pandas as pd
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import  train_test_split
from sklearn.preprocessing import StandardScaler

url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"

columns = ["Pregnancies",
           'Glucose',
           'BloodPressure',
           'SkinThickness',
           'Insulin',
           'BMI',
           'DiabetesPedigreeFunction',
           'age',
           'Outcome']
df = pd.read_csv(url, names=columns, skiprows=1)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Preprocessing Using SMOTE

The class distribution confirms the imbalance. To handle this, we can use an oversampling technique like SMOTE (Synthetic Minority Over-sampling Technique) to create synthetic samples for the minority class.

In [3]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
age,0
Outcome,0


In [4]:
import numpy as np
cols_with_missing_vals = ['Glucose',"BloodPressure","Insulin","BMI"]
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0,np.nan) ## Converting all the zero values into nan Values
df.fillna(df.mean(),inplace = True) ## Converting all the Nan Values with their mean values

In [5]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
age,0
Outcome,0


In [6]:
X = df.drop('Outcome',axis = 1)
y = df['Outcome']

In [7]:
X_train,x_test,y_train,y_test = train_test_split(X,y,test_size= 0.2,random_state = 42)

In [8]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

print('Class distribution after SMOTE:')
display(y_train.value_counts())

Class distribution after SMOTE:


,count
Outcome,
0,401
1,401


In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_test = scaler.transform(x_test)

## Applying the Model using HyperParameter Tuning using Optuna

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def objective(trail):
  ''' Creating the Required Values of both the Hyper Parameters'''
  n_estimators = trail.suggest_int('n_estimators',50,300) ## Values of N-Estimators Ranges Between 50 and 300
  max_depth = trail.suggest_int('max_depth',3,20) ## Values of Max-Depth Ranges between 3 and 20

  ''' Creating a Random Forest Classifier with the suggested Parameters'''
  ''' Basically Creating a WorkSpace'''
  model = RandomForestClassifier(
      n_estimators= n_estimators,
      max_depth = max_depth,
      random_state = 42
  )
  ''' Calculating the Cross Validation Score'''
  score = cross_val_score(model,X_train,y_train,cv = 3,scoring = 'accuracy').mean() ## Multiple Accracy Scores's Mean
  return score

In [11]:
''' Creating a New Study'''

'''Using the Sampler as Tree Parzen Estimator''' ## (By Default Bayseian Search)
study = optuna.create_study(
    direction = 'maximize', ## Helps in identifying the next Set of Points
    sampler = optuna.samplers.TPESampler(),
)

study.optimize(objective, n_trials = 50)

[I 2026-06-30 05:59:31,036] A new study created in memory with name: no-name-d9801560-ad9e-4590-ad88-1b52c6ee74a3
[I 2026-06-30 05:59:33,025] Trial 0 finished with value: 0.8291827379954162 and parameters: {'n_estimators': 233, 'max_depth': 20}. Best is trial 0 with value: 0.8291827379954162.
[I 2026-06-30 05:59:34,048] Trial 1 finished with value: 0.8217060763597742 and parameters: {'n_estimators': 62, 'max_depth': 18}. Best is trial 0 with value: 0.8291827379954162.
[I 2026-06-30 05:59:35,870] Trial 2 finished with value: 0.824193638548829 and parameters: {'n_estimators': 93, 'max_depth': 13}. Best is trial 0 with value: 0.8291827379954162.
[I 2026-06-30 05:59:38,290] Trial 3 finished with value: 0.8242029552611475 and parameters: {'n_estimators': 150, 'max_depth': 18}. Best is trial 0 with value: 0.8291827379954162.
[I 2026-06-30 05:59:43,386] Trial 4 finished with value: 0.8167169769131869 and parameters: {'n_estimators': 237, 'max_depth': 7}. Best is trial 0 with value: 0.82918273

In [12]:
print(f"Best Trial Parameters: {study.best_trial.params}")
print(f"Best Trial Value: {study.best_trial.value}")

Best Trial Parameters: {'n_estimators': 138, 'max_depth': 16}
Best Trial Value: 0.8329327147036354


## Evaluation

In [13]:
from sklearn.metrics import accuracy_score,classification_report
bestModel = RandomForestClassifier(
    n_estimators= study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)

In [14]:
bestModel.fit(X_train,y_train)

RandomForestClassifier(max_depth=16, n_estimators=138, random_state=42)

In [15]:
y_pred = bestModel.predict(x_test)

In [16]:
testAccuracy = accuracy_score(y_test,y_pred)
print(f"Test Accuracy: {testAccuracy}")

Test Accuracy: 0.7662337662337663


## Samplers in Optuna

#### Building the Random Search Using Optuna Without Sklearn

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def objective(trail):
  ''' Creating the Required Values of both the Hyper Parameters'''
  n_estimators = trail.suggest_int('n_estimators',50,300) ## Values of N-Estimators Ranges Between 50 and 300
  max_depth = trail.suggest_int('max_depth',3,20) ## Values of Max-Depth Ranges between 3 and 20

  ''' Creating a Random Forest Classifier with the suggested Parameters'''
  ''' Basically Creating a WorkSpace'''
  model = RandomForestClassifier(
      n_estimators= n_estimators,
      max_depth = max_depth,
      random_state = 42
  )
  ''' Calculating the Cross Validation Score'''
  score = cross_val_score(model,X_train,y_train,cv = 3,scoring = 'accuracy').mean() ## Multiple Accracy Scores's Mean
  return score

In [18]:
''' Creating a New Study'''

'''Using the Sampler as Tree Parzen Estimator''' ## (By Default Bayseian Search)
study = optuna.create_study(
    direction = 'maximize', ## Helps in identifying the next Set of Points
    sampler = optuna.samplers.RandomSampler(),
)

study.optimize(objective, n_trials = 50)

[I 2026-06-30 06:01:51,188] A new study created in memory with name: no-name-a1c92d9d-c851-4992-9918-f8dbee9349b1
[I 2026-06-30 06:01:51,865] Trial 0 finished with value: 0.7817746473624387 and parameters: {'n_estimators': 55, 'max_depth': 3}. Best is trial 0 with value: 0.7817746473624387.
[I 2026-06-30 06:01:53,343] Trial 1 finished with value: 0.8217060763597742 and parameters: {'n_estimators': 117, 'max_depth': 15}. Best is trial 1 with value: 0.8217060763597742.
[I 2026-06-30 06:01:57,624] Trial 2 finished with value: 0.8304311774461027 and parameters: {'n_estimators': 292, 'max_depth': 19}. Best is trial 2 with value: 0.8304311774461027.
[I 2026-06-30 06:02:00,420] Trial 3 finished with value: 0.8266951758063614 and parameters: {'n_estimators': 208, 'max_depth': 19}. Best is trial 2 with value: 0.8304311774461027.
[I 2026-06-30 06:02:02,081] Trial 4 finished with value: 0.8167123185570276 and parameters: {'n_estimators': 247, 'max_depth': 8}. Best is trial 2 with value: 0.8304311

In [19]:
from sklearn.metrics import accuracy_score,classification_report
bestModel = RandomForestClassifier(
    n_estimators= study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)
bestModel.fit(X_train,y_train)
y_pred = bestModel.predict(x_test)
testAccuracy = accuracy_score(y_test,y_pred)
print(f"Test Accuracy: {testAccuracy}")

Test Accuracy: 0.7662337662337663


#### Building the Grid| Search Using Optuna Without Sklearn

In [20]:
search_space = {
    'n_estimators': [50, 100, 150, 200, 250, 300],
    'max_depth': [3, 5, 7, 9]
}

In [22]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.GridSampler(search_space)
)
study.optimize(objective)

[I 2026-06-30 06:12:18,536] A new study created in memory with name: no-name-784c3902-8164-45b0-a38b-581af62e44b0
[I 2026-06-30 06:12:22,003] Trial 0 finished with value: 0.8042279240501612 and parameters: {'n_estimators': 300, 'max_depth': 5}. Best is trial 0 with value: 0.8042279240501612.
[I 2026-06-30 06:12:24,979] Trial 1 finished with value: 0.8042279240501612 and parameters: {'n_estimators': 250, 'max_depth': 5}. Best is trial 0 with value: 0.8042279240501612.
[I 2026-06-30 06:12:30,214] Trial 2 finished with value: 0.8204483201967689 and parameters: {'n_estimators': 250, 'max_depth': 9}. Best is trial 2 with value: 0.8204483201967689.
[I 2026-06-30 06:12:32,100] Trial 3 finished with value: 0.8167123185570276 and parameters: {'n_estimators': 150, 'max_depth': 7}. Best is trial 2 with value: 0.8204483201967689.
[I 2026-06-30 06:12:33,893] Trial 4 finished with value: 0.8204483201967689 and parameters: {'n_estimators': 150, 'max_depth': 9}. Best is trial 2 with value: 0.820448320

In [23]:
from sklearn.metrics import accuracy_score,classification_report
bestModel = RandomForestClassifier(
    n_estimators= study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)
bestModel.fit(X_train,y_train)
y_pred = bestModel.predict(x_test)
testAccuracy = accuracy_score(y_test,y_pred)
print(f"Test Accuracy: {testAccuracy}")

Test Accuracy: 0.7727272727272727


## Optuna Visualizations

In [24]:
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_contour,plot_param_importances
plot_optimization_history(study).show()
''' Graph Bteweem the Objective Value and Trilas'''

' Graph Bteweem the Objective Value and Trilas'

In [25]:
plot_parallel_coordinate(study).show()

In [26]:
plot_slice(study).show()

In [27]:
plot_contour(study).show()

In [28]:
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [29]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [30]:
def objective(trial):
  classifier = trial.suggest_categorical('classifier',['SVM','RandomForest','GradientBoosting','LogisticRegression'])

  if classifier == 'SVM':
    c = trial.suggest_float('C',0.1,100,log = True)
    kernel = trial.suggest_categorical('kernel',['linear','rbf','poly','sigmoid'])
    gamma = trial.suggest_categorical('gamma',['scale','auto'])
    model = SVC(C = c,kernel = kernel,gamma = gamma,random_state = 42)

  elif classifier == 'RandomForest':
    nestimators = trial.suggest_int('n_estimators',50,300)
    max_depth = trial.suggest_int('max_depth',3,20)
    min_split = trial.suggest_int('min_split',2,10)
    min_leaf = trial.suggest_int('min_leaf',1,10)
    model = RandomForestClassifier(
        n_estimators = nestimators,
        max_depth = max_depth,
        min_samples_split=min_split,
        min_samples_leaf=min_leaf,
        random_state = 42
    )

  elif classifier == 'GradientBoosting':
    n_estimators = trial.suggest_int('n_estimators',50,300) # Corrected typo from nestimaotrs
    learning_rate = trial.suggest_float('learning_rate',0.001, 0.3) # Added high argument
    max_depth = trial.suggest_int('max_depth',3,20)
    min_split = trial.suggest_int('min_split',2,10)
    min_leaf = trial.suggest_int('min_leaf',1,1) # min_leaf usually has a range, might want to check this
    model = GradientBoostingClassifier(
        n_estimators = n_estimators,
        learning_rate = learning_rate,
        max_depth = max_depth,
        min_samples_split=min_split,
        min_samples_leaf=min_leaf
    )

  else :
    c = trial.suggest_float('C',0.1,100,log = True)
    max_iter = trial.suggest_int('max_iter',100,1000)
    model = LogisticRegression(
        C = c,
        max_iter = max_iter,
        random_state = 42
    )

  score = cross_val_score(model,X_train,y_train,cv = 3,scoring = 'accuracy').mean()
  return score

In [31]:
study = optuna.create_study(direction= 'maximize')
study.optimize(objective,n_trials = 100)

[I 2026-06-30 06:13:29,080] A new study created in memory with name: no-name-c671e02f-9f59-42ad-b942-8d78cfba7cfb
[I 2026-06-30 06:13:32,452] Trial 0 finished with value: 0.7506568282184584 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 168, 'learning_rate': 0.29215512295243007, 'max_depth': 20, 'min_split': 4, 'min_leaf': 1}. Best is trial 0 with value: 0.7506568282184584.
[I 2026-06-30 06:13:37,134] Trial 1 finished with value: 0.8192138558145601 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 250, 'learning_rate': 0.05092108706861579, 'max_depth': 7, 'min_split': 5, 'min_leaf': 1}. Best is trial 1 with value: 0.8192138558145601.
[I 2026-06-30 06:13:37,160] Trial 2 finished with value: 0.7493385134253825 and parameters: {'classifier': 'LogisticRegression', 'C': 42.66146500664712, 'max_iter': 556}. Best is trial 1 with value: 0.8192138558145601.
[I 2026-06-30 06:13:37,249] Trial 3 finished with value: 0.7580915646486668 and parameters: {'classifi

In [32]:
print(f"Best Trial Parameters: {study.best_trial.params}")
print(f"Best Trial Value: {study.best_trial.value}")

Best Trial Parameters: {'classifier': 'GradientBoosting', 'n_estimators': 188, 'learning_rate': 0.29718284552640606, 'max_depth': 17, 'min_split': 7, 'min_leaf': 1}
Best Trial Value: 0.8441360612667003


In [33]:
from sklearn.metrics import accuracy_score,classification_report
bestModel = RandomForestClassifier(
    n_estimators= study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)
bestModel.fit(X_train,y_train)
y_pred = bestModel.predict(x_test)
testAccuracy = accuracy_score(y_test,y_pred)
print(f"Test Accuracy: {testAccuracy}")

Test Accuracy: 0.7727272727272727


In [34]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_max_iter,params_min_leaf,params_min_split,params_n_estimators,state
0,0,0.750657,2026-06-30 06:13:29.083505,2026-06-30 06:13:32.452839,0 days 00:00:03.369334,NaN,GradientBoosting,NaN,NaN,0.292155,20.0,NaN,1.0,4.0,168.0,COMPLETE
1,1,0.819214,2026-06-30 06:13:32.454100,2026-06-30 06:13:37.134274,0 days 00:00:04.680174,NaN,GradientBoosting,NaN,NaN,0.050921,7.0,NaN,1.0,5.0,250.0,COMPLETE
2,2,0.749339,2026-06-30 06:13:37.135547,2026-06-30 06:13:37.159970,0 days 00:00:00.024423,42.661465,LogisticRegression,NaN,NaN,NaN,NaN,556.0,NaN,NaN,NaN,COMPLETE
3,3,0.758092,2026-06-30 06:13:37.162029,2026-06-30 06:13:37.249738,0 days 00:00:00.087709,1.410402,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.749339,2026-06-30 06:13:37.250820,2026-06-30 06:13:37.273830,0 days 00:00:00.023010,95.100083,LogisticRegression,NaN,NaN,NaN,NaN,864.0,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.826691,2026-06-30 06:18:41.603455,2026-06-30 06:18:46.936248,0 days 00:00:05.332793,NaN,GradientBoosting,NaN,NaN,0.183507,17.0,NaN,1.0,8.0,198.0,COMPLETE
96,96,0.824184,2026-06-30 06:18:46.937446,2026-06-30 06:18:52.092154,0 days 00:00:05.154708,NaN,GradientBoosting,NaN,NaN,0.157411,19.0,NaN,1.0,7.0,189.0,COMPLETE
97,97,0.760579,2026-06-30 06:18:52.093407,2026-06-30 06:18:52.152305,0 days 00:00:00.058898,3.725428,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.749339,2026-06-30 06:18:52.153707,2026-06-30 06:18:52.185476,0 days 00:00:00.031769,1.031408,LogisticRegression,NaN,NaN,NaN,NaN,329.0,NaN,NaN,NaN,COMPLETE


In [35]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
GradientBoosting,73
LogisticRegression,10
SVM,9
RandomForest,8


In [36]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.822113
LogisticRegression,0.748839
RandomForest,0.815157
SVM,0.747004
